Task 1 si 2

In [12]:
import nltk

grammar = nltk.CFG.fromstring("""
S       -> NP VP

NP      -> Det Nominal | Nominal | VPger | NP PP
Nominal -> Noun | Adj Nominal | Nominal Conj Nominal
VPger   -> Ger NP

VP      -> V NP | V Adv | Aux VP | Modal VP | Cop Adj | VP PP
PP      -> Prep NP

Det     -> 'the'
Noun    -> 'professors' | 'men' | 'women' | 'reporter' | 'actor' | 'camera' | 'entrance'
Adj     -> 'visiting' | 'old' | 'annoying'
Ger     -> 'visiting'
Modal   -> 'can'
Cop     -> 'be'
Aux     -> 'were'
V       -> 'waiting' | 'interviewed'
Adv     -> 'outside'
Conj    -> 'and'
Prep    -> 'with' | 'near'
""")

sentences = [
    "Visiting professors can be annoying.",
    "Old men and women were waiting outside.",
    "The reporter interviewed the actor with the camera near the entrance.",
]

def tokenize(s):
    return s.lower().rstrip(".").split()

for parser_cls in (nltk.ChartParser, nltk.EarleyChartParser):
    parser = parser_cls(grammar)
    print("=" * 70)
    print(parser_cls.__name__)
    for s in sentences:
        trees = list(parser.parse(tokenize(s)))
        print(f"\n{s}  ->  {len(trees)} arbori")
        for i, t in enumerate(trees, 1):
            print(f"--- arbore {i}")
            print(t)

ChartParser

Visiting professors can be annoying.  ->  2 arbori
--- arbore 1
(S
  (NP (Nominal (Adj visiting) (Nominal (Noun professors))))
  (VP (Modal can) (VP (Cop be) (Adj annoying))))
--- arbore 2
(S
  (NP (VPger (Ger visiting) (NP (Nominal (Noun professors)))))
  (VP (Modal can) (VP (Cop be) (Adj annoying))))

Old men and women were waiting outside.  ->  2 arbori
--- arbore 1
(S
  (NP
    (Nominal
      (Nominal (Adj old) (Nominal (Noun men)))
      (Conj and)
      (Nominal (Noun women))))
  (VP (Aux were) (VP (V waiting) (Adv outside))))
--- arbore 2
(S
  (NP
    (Nominal
      (Adj old)
      (Nominal
        (Nominal (Noun men))
        (Conj and)
        (Nominal (Noun women)))))
  (VP (Aux were) (VP (V waiting) (Adv outside))))

The reporter interviewed the actor with the camera near the entrance.  ->  5 arbori
--- arbore 1
(S
  (NP (Det the) (Nominal (Noun reporter)))
  (VP
    (V interviewed)
    (NP
      (NP
        (NP (Det the) (Nominal (Noun actor)))
        (PP (Pre

Task 3. (0.25p) Implement a dependency parser (using, for instance, spaCY, NLTK or Stanza) and
parse the three sentences above.

In [11]:
import spacy
from spacy import displacy


nlp = spacy.load("en_core_web_sm")

sentences = [
    "Visiting professors can be annoying.",
    "Old men and women were waiting outside.",
    "The reporter interviewed the actor with the camera near the entrance."
]

for i, sent_text in enumerate(sentences, 1):
    print(f"\n--- Propoziția {i}: \"{sent_text}\" ---")
    doc = nlp(sent_text)

    #(Head -> Dependent [Label])
    print(f"{'Cuvânt (Dependent)':<15} | {'Relație (DEPREL)':<15} | {'Head (Cuvântul Guvernator)'}")
    print("-" * 55)
    for token in doc:
        print(f"{token.text:<15} | {token.dep_:<15} | {token.head.text}")




--- Propoziția 1: "Visiting professors can be annoying." ---
Cuvânt (Dependent) | Relație (DEPREL) | Head (Cuvântul Guvernator)
-------------------------------------------------------
Visiting        | amod            | professors
professors      | nsubj           | be
can             | aux             | be
be              | ROOT            | be
annoying        | acomp           | be
.               | punct           | be

--- Propoziția 2: "Old men and women were waiting outside." ---
Cuvânt (Dependent) | Relație (DEPREL) | Head (Cuvântul Guvernator)
-------------------------------------------------------
Old             | amod            | men
men             | nsubj           | waiting
and             | cc              | men
women           | conj            | men
were            | aux             | waiting
waiting         | ROOT            | waiting
outside         | advmod          | waiting
.               | punct           | waiting

--- Propoziția 3: "The reporter interviewed 

Task 4. (0,30p) Write a program which transforms a Context Free Grammar, as the one discussed in
class,
into a Chomsky Normal Form grammar, also converting the probabilities. Test it on the
grammar from the course.

In [1]:
from collections import defaultdict
import nltk
from nltk import PCFG
from nltk.grammar import Nonterminal, ProbabilisticProduction

# Definim gramatica probabilistică independentă de context (PCFG) din curs.
# Fiecare regulă are forma: ParteaStângă -> ParteaDreaptă [Probabilitate]
course_pcfg = PCFG.fromstring("""
    S -> NP VP [0.8]
    S -> Aux NP VP [0.1]
    S -> VP [0.1]

    NP -> Pronoun [0.2]
    NP -> Proper_Noun [0.2]
    NP -> Det Nominal [0.6]

    Nominal -> Noun [0.3]
    Nominal -> Nominal Noun [0.2]
    Nominal -> Nominal PP [0.5]

    VP -> Verb [0.2]
    VP -> Verb NP [0.5]
    VP -> VP PP [0.3]

    PP -> Prep NP [1.0]

    Det -> 'the' [0.6] | 'a' [0.2] | 'that' [0.1] | 'this' [0.1]
    Noun -> 'book' [0.3] | 'flight' [0.3] | 'meal' [0.2] | 'money' [0.2]
    Pronoun -> 'I' [0.25] | 'he' [0.25] | 'she' [0.25] | 'me' [0.25]
    Proper_Noun -> 'Houston' [0.8] | 'NWA' [0.2]
    Verb -> 'book' [0.3] | 'include' [0.4] | 'prefer' [0.3]
    Aux -> 'does' [1.0]
    Prep -> 'from' [0.25] | 'to' [0.25] | 'on' [0.1] | 'near' [0.2] | 'through' [0.2]
""")


def convert_pcfg_to_cnf(pcfg):
  """Convertește o gramatică PCFG în Forma Normală Chomsky (CNF).

  Regulile CNF permise sunt doar:
  1. A -> B C (două nonterminale în dreapta)
  2. A -> 'cuvânt' (un singur terminal în dreapta)
  """
  # Contor pentru a crea nume unice variabilelor auxiliare (ex: X1, X2...)
  counter = 1

  # Extragerea tuturor regulilor (producțiilor) din gramatica inițială
  prods = list(pcfg.productions())


  # PASUL 1: Izolarea terminalelor mixte (TERM)
  # În CNF, o regulă cu mai mult de un simbol în dreapta NU are voie să conțină cuvinte direct.
  # Exemplu: Dacă avem o regulă de tipul A -> 'the' B, cuvântul 'the' este înlocuit cu T_THE.
  step1 = []
  term_nts = {}  # Dicționar pentru a păstra corespondența dintre cuvânt și noul nonterminal (ex: 'the' -> T_THE)

  for p in prods:
    rhs = p.rhs()  # Partea dreaptă a regulii (Right-Hand Side)

    # Dacă regula are mai mult de 1 simbol în dreapta (RHS > 1)
    if len(rhs) > 1:
      new_rhs = []
      for sym in rhs:
        # Dacă simbolul este un cuvânt/terminal (nu este instanță de Nonterminal)
        if not isinstance(sym, Nonterminal):
          # Dacă nu am creat deja un nonterminal pentru acest cuvânt, îl creăm acum
          if sym not in term_nts:
            term_nts[sym] = Nonterminal(f"T_{str(sym).upper()}")
          new_rhs.append(term_nts[sym])
        else:
          # Dacă este deja un nonterminal, îl lăsăm neschimbat
          new_rhs.append(sym)

      # Salvăm noua regulă modificată (fără terminale mixte în dreapta)
      step1.append(ProbabilisticProduction(p.lhs(), new_rhs, prob=p.prob()))
    else:
      # Regulile de lungime 1 (sau deja corecte) trec direct mai departe
      step1.append(p)

  # Adăugăm în lista de reguli noile producții terminale create (ex: T_THE -> 'the' [1.0])
  for t, nt in term_nts.items():
    step1.append(ProbabilisticProduction(nt, [t], prob=1.0))


  # PASUL 2: Binarizarea regulilor lungi (BIN)
  # În CNF, o regulă cu nonterminale în dreapta poate avea MAXIMUM 2 simboluri.
  # Exemplu: S -> Aux NP VP devine S -> Aux X1 [prob_originală] și X1 -> NP VP [1.0]

  step2 = []
  for p in step1:
    rhs = p.rhs()

    # Dacă partea dreaptă are mai mult de 2 simboluri
    if len(rhs) > 2:
      current_lhs = p.lhs()  # Salvează simbolic din stânga (ex: S)
      prob = p.prob()  # Salvează probabilitatea inițială a regulii

      # Sparge regula lungă în perechi succesive de câte 2 simboluri
      for i in range(len(rhs) - 2):
        new_nt = Nonterminal(f"X{counter}")  # Creează o variabilă intermediară: X1, X2...
        counter += 1

        # Prima regulă din lanț primește probabilitatea originală a regulii lungi
        step2.append(
            ProbabilisticProduction(current_lhs, [rhs[i], new_nt], prob=prob)
        )

        current_lhs = new_nt  # Noul element din stânga devine simbolul intermediar
        prob = 1.0  # Regulile intermediare rezultate vor avea probabilitatea 1.0

      # Adaugă ultima pereche din lanțul binarizat
      step2.append(
          ProbabilisticProduction(
              current_lhs, [rhs[-2], rhs[-1]], prob=1.0
          )
      )
    else:
      # Regulile care au deja 1 sau 2 simboluri în dreapta trec neschimbate
      step2.append(p)

  # PASUL 3: Eliminarea regulilor unitare (UNIT) - de forma A -> B (unde B e nonterminal)
  # În CNF strict, nu avem voie să avem reguli de la un singur nonterminal la altul.

  # Separăm regulile unitare de cele non-unitare (binare sau terminale)
  unit = defaultdict(dict)  # Dicționar: unit[A][B] = probabilitatea regulii A -> B
  non_unit = []

  for p in step2:
    # O regulă e unitară dacă are un singur simbol în dreapta și acel simbol e un Nonterminal
    if len(p.rhs()) == 1 and isinstance(p.rhs()[0], Nonterminal):
      unit[p.lhs()][p.rhs()[0]] = p.prob()
    else:
      non_unit.append(p)

  # Funcție ajutătoare: Calculează închiderea tranzitivă pentru regulile unitare.
  # Dacă A -> B (p1) și B -> C (p2), atunci A -> C cu probabilitatea p1 * p2.
  def closure(A):
    result = {A: 1.0}  # Se începe de la starea A -> A cu probabilitatea 1.0
    stack = [A]

    while stack:
      X = stack.pop()
      # Căutăm toate regulile unitare care pornesc din X (X -> Y)
      for Y, pr in unit.get(X, {}).items():
        new_p = result[X] * pr  # Înmulțim probabilitățile pe calea de derivare
        # Dacă găsim o cale cu o probabilitate mai mare către Y, o actualizăm
        if new_p > result.get(Y, 0.0):
          result[Y] = new_p
          stack.append(Y)
    return result

  # Grupăm regulile non-unitare după simbolul din stânga lor (LHS)
  by_lhs = defaultdict(list)
  for p in non_unit:
    by_lhs[p.lhs()].append(p)

  # Extragerea tuturor simbolurilor nonterminale existente
  all_lhs = set(p.lhs() for p in step2)
  final = {}  # Dicționar pentru stocarea regulilor finale re-calculate

  # Pentru fiecare nonterminal A
  for A in all_lhs:
    # Aflăm toate nodurile B la care poate ajunge A prin reguli unitare (A =>* B)
    for B, pr_AB in closure(A).items():
      # Luăm toate producțiile valide (non-unitare) ale lui B și i le atribuim direct lui A
      for p in by_lhs.get(B, []):
        key = (A, p.rhs())
        # Probabilitatea totală este suma probabilităților tuturor căilor posibile
        final[key] = final.get(key, 0.0) + pr_AB * p.prob()

  # Construim obiectele de tip ProbabilisticProduction finale
  final_prods = [
      ProbabilisticProduction(lhs, list(rhs), prob=pr)
      for (lhs, rhs), pr in final.items()
  ]

  # Returnăm noua gramatică re-sintetizată în format CNF
  return PCFG(pcfg.start(), final_prods)


# =========================================================================
# EXECUTAREA ȘI AFIȘAREA REZULTATELOR
# =========================================================================

# Apelăm funcția de conversie
cnf_pcfg = convert_pcfg_to_cnf(course_pcfg)

# Afișăm gramatica rezultată
print("=== GRAMATICA CONVERTITĂ ÎN CNF ===")
for production in cnf_pcfg.productions():
  print(production)

=== GRAMATICA CONVERTITĂ ÎN CNF ===
Pronoun -> 'I' [0.25]
Pronoun -> 'he' [0.25]
Pronoun -> 'she' [0.25]
Pronoun -> 'me' [0.25]
Det -> 'the' [0.6]
Det -> 'a' [0.2]
Det -> 'that' [0.1]
Det -> 'this' [0.1]
Prep -> 'from' [0.25]
Prep -> 'to' [0.25]
Prep -> 'on' [0.1]
Prep -> 'near' [0.2]
Prep -> 'through' [0.2]
X1 -> NP VP [1.0]
Nominal -> Nominal Noun [0.2]
Nominal -> Nominal PP [0.5]
Nominal -> 'book' [0.09]
Nominal -> 'flight' [0.09]
Nominal -> 'meal' [0.06]
Nominal -> 'money' [0.06]
Verb -> 'book' [0.3]
Verb -> 'include' [0.4]
Verb -> 'prefer' [0.3]
Proper_Noun -> 'Houston' [0.8]
Proper_Noun -> 'NWA' [0.2]
S -> NP VP [0.8]
S -> Aux X1 [0.1]
S -> Verb NP [0.05]
S -> VP PP [0.03]
S -> 'book' [0.006]
S -> 'include' [0.008]
S -> 'prefer' [0.006]
PP -> Prep NP [1.0]
Aux -> 'does' [1.0]
Noun -> 'book' [0.3]
Noun -> 'flight' [0.3]
Noun -> 'meal' [0.2]
Noun -> 'money' [0.2]
VP -> Verb NP [0.5]
VP -> VP PP [0.3]
VP -> 'book' [0.06]
VP -> 'include' [0.08]
VP -> 'prefer' [0.06]
NP -> Det Nominal

5. (0,15p) Describe an application n which constituency and/or dependency parsing can provide
useful information. Explain briefly why. (1/2 pag.)

An application where syntactic parsing provides useful information is relation extraction from text, for example, the automated construction of a knowledge base from news articles, starting from sentences such as "Old men and women were waiting outside."

The role of dependency parsing. A dependency parser directly identifies the grammatical relations between words: the nominal subject (nsubj), the conjunct (conj), coordination (cc), auxiliaries, and modifiers. For the example above, we obtain relations showing that both "men" and "women" act as the subjects of the action "waiting". These grammatical triplets and dependencies can be extracted independently of word order or sentence length, which a simple keyword or position-based search cannot achieve. Similarly, in passive sentences or sentences with long intervening modifiers, the logical subject remains easy to retrieve from the dependency tree.

The role of constituency parsing. The phrase structure tree delimits noun phrases, verb phrases, and prepositional phrases (NP, VP, PP), thus allowing the extraction of complete entities rather than just isolated words. Furthermore, the tree structure reveals scope and attachment ambiguities: in the given sentence, "Old men and women" has two distinct phrase structure trees. In one structure, the adjective "Old" modifies only "men" ([NP [NP Old men] and [NP women]]), meaning the women are not necessarily old. In the second structure, "Old" modifies the entire coordinated phrase ([NP Old [NP men and women]]), meaning both the men and the women are old. A system capable of distinguishing these trees can choose the correct interpretation or signal the structural ambiguity. Likewise, "Visiting professors can be annoying" has two parses (professors who visit are annoying / visiting professors is annoying), and "The reporter interviewed the actor with the camera" can attach the prepositional phrase to either the verb or the noun.

Why it is useful. Without syntactic information, a bag-of-words system cannot distinguish "old men and women" from "women and old men" or properly determine the scope of modifiers and relations. Syntactic parsing provides precisely this structure, upon which applications such as question answering, machine translation (word reordering across languages), sentiment analysis (e.g., the scope of negation or adjectives), and automatic summarization depend. In practice, dependency parsing is preferred for relation extraction because it is more compact and easier to use on real-world data, whereas constituency parsing is useful when the complete delimitation of phrases and the handling of structural scope ambiguities matter.